# IndiVoice-DeepASR: Cloud Gateway 🚀

This notebook is your command center for training the **IndiVoice-DeepASR** model using Google Colab GPUs. 

### 📁 Cloud-to-Cloud Workflow
This notebook is configured to download datasets directly from **Hugging Face** and **GitHub** into your Google Drive for persistent storage.

## 1. Environment & Drive Initialization

In [ ]:
# 1. Verify GPU
!nvidia-smi

# 2. Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 3. Configure Paths
DRIVE_BASE = '/content/drive/MyDrive/IndiVoice-DeepASR'
os.makedirs(DRIVE_BASE, exist_ok=True)

# 4. Sync Repository
%cd "{DRIVE_BASE}"
if not os.path.exists('.git'):
    !git clone https://github.com/purvanshjoshi/IndiVoice-DeepASR.git .
else:
    !git fetch --all
    !git reset --hard origin/main

# 5. Install Dependencies
!pip install -r requirements.txt

print("\n✅ Setup Complete. All scripts (src/) are verified in Drive.")

## 2. Hugging Face Authentication (Required for Svarah)
The Svarah dataset is **gated**. Follow these steps first:
1. Visit [huggingface.co/datasets/ai4bharat/Svarah](https://huggingface.co/datasets/ai4bharat/Svarah) and click **'Agree and access repository'** (you'll need a free HF account).
2. Get your Access Token from [hf.co/settings/tokens](https://hf.co/settings/tokens).
3. Run the cell below and paste your token.

In [ ]:
from huggingface_hub import login
login()

## 3. Cloud Data Acquisition
Run this cell to download the research datasets (Svarah, IndicAccentDb, etc.) directly into your Google Drive.

In [ ]:
# Downloads datasets directly to data/raw in your Drive
!python src/download_data.py --data_dir data/raw

## 4. Phase 1: Preprocessing
Standardizes audio and generates the manifest for training. Choose your dataset below.

In [ ]:
# OPTION A: Preprocess Svarah (Hugging Face Dataset)
!python src/preprocess.py \
    --hf_dataset ai4bharat/Svarah \
    --output_dir data/processed/svarah \
    --manifest_path data/processed/svarah_manifest.json

# OPTION B: Preprocess IndicAccentDB (Hugging Face Dataset)
# !python src/preprocess.py \
#     --hf_dataset DarshanaS/IndicAccentDb \
#     --output_dir data/processed/indic_accent \
#     --manifest_path data/processed/indic_manifest.json

## 5. Phase 2: Deep Learning Training
Fine-tunes Whisper-Medium using LoRA. Best checkpoints are saved back to your Google Drive.

In [ ]:
!python src/train.py \
    --train_manifest data/processed/svarah_manifest.json \
    --val_manifest data/processed/svarah_manifest.json \
    --output_dir models/whisper-indian-lora

## 6. Phase 3: Evaluation
Test the performance on unseen data.

In [ ]:
!python src/evaluate.py \
    --model_path models/whisper-indian-lora/final \
    --test_manifest data/processed/svarah_manifest.json

## 7. Phase 4: Launch Demo
Starts a Gradio web app. Click the public `gradio.live` link to try your model.

In [ ]:
!python src/deploy.py --model_path models/whisper-indian-lora/final --share